# Automatic Plant Watering System (KNN) — Fixed Version
**What was fixed compared to the original notebook:**
1. The `scaler` was fitted twice (the second one overwrote the first), so the prediction in the last cell was wrong. Now the imputer, scaler, and model all live in **one Pipeline**.
2. The imputer and scaler were applied to the whole dataset before the split (data leakage), and the imputer also saw the `Status` column (the target). Now the split happens first, fitting is done on the training set only, and `Status` is not passed to the imputer.
3. The column name `" Soil Humidity"` had a leading space, which was removed.
4. Added model export with joblib and a test (simulation) section that loads the saved file and runs predictions on it.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib, sklearn

from sklearn.preprocessing import LabelBinarizer, MinMaxScaler
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 1) Load the data

In [ ]:
df = pd.read_csv('/content/TARP.csv.zip')
df.columns = df.columns.str.strip()   # the " Soil Humidity" column had a leading space
df.head()

In [ ]:
encoder = LabelBinarizer()
df['Status'] = encoder.fit_transform(df['Status']).ravel()
print("classes:", encoder.classes_, "-> 0/1")
df.info()

In [ ]:
df.describe().T

## 2) Explore the data

In [ ]:
plt.figure(figsize=(14, 14))
sns.heatmap(df.corr(), annot=True, fmt=".2f")
plt.show()

In [ ]:
for col in df.columns[:-1]:
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(col)
    plt.show()

In [ ]:
g = sns.clustermap(df.corr(), method='complete', cmap='RdBu', annot=True, annot_kws={'size': 8})
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=60)
plt.show()

In [ ]:
plt.hist(df['Status'])
plt.title("Status distribution")
plt.show()

## 3) Split first (before any imputer or scaler)

In [ ]:
FEATURES = [c for c in df.columns if c != 'Status']
X = df[FEATURES]
y = df['Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=7, stratify=y
)
print(X_train.shape, X_test.shape)

## 4) Model: Pipeline (imputer → scaler → KNN)

In [ ]:
pipe = Pipeline([
    ("imputer", IterativeImputer(random_state=46)),
    ("scaler",  MinMaxScaler((-1, 1))),
    ("knn",     KNeighborsClassifier(n_neighbors=3)),
])
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

## 5) Export the model (joblib)

In [ ]:
bundle = {
    "pipeline": pipe,                      # imputer + scaler + KNN
    "features": FEATURES,                  # column names and order
    "classes": list(encoder.classes_),     # ['OFF', 'ON'] -> index 0/1
    "sklearn_version": sklearn.__version__,
}
joblib.dump(bundle, "watering_knn.joblib", compress=3)
print("saved: watering_knn.joblib")

try:
    from google.colab import files
    files.download("watering_knn.joblib")
except ImportError:
    pass

## 6) Test / Simulation
Loads the joblib file from disk (the same way it will be loaded in the project) and runs predictions on it. Any missing sensor is left as `NaN`, and the imputer inside the pipeline fills it in.

In [ ]:
bundle = joblib.load("watering_knn.joblib")

def predict_status(reading: dict):
    """reading: dict keyed by column name. Any missing column is set to NaN and imputed automatically."""
    row = pd.DataFrame([{f: reading.get(f, np.nan) for f in bundle["features"]}])
    proba = bundle["pipeline"].predict_proba(row)[0]
    idx = int(np.argmax(proba))
    return bundle["classes"][idx], float(proba[idx])

In [ ]:
# Test 1: the original example (all 14 values)
input_data = (5,23,27,4,21.44,5.8,28.38,22.67,101.46,7.375482851,224.0581164,98,47,37)
sample = dict(zip(bundle["features"], input_data))
label, conf = predict_status(sample)
print(f"Pump: {label}  (confidence {conf:.2f})")

In [ ]:
# Test 2: only the ESP32 sensors, everything else missing
partial = {"Soil Moisture": 20, "Temperature": 30, "Soil Humidity": 35, "Time": 60}
print(predict_status(partial))

In [ ]:
# Test 3: 10 random rows from the test set compared with the ground truth
sim = X_test.sample(10, random_state=1)
rows = []
for idx, r in sim.iterrows():
    label, conf = predict_status(r.to_dict())
    rows.append({"Soil Moisture": r["Soil Moisture"],
                 "predicted": label, "confidence": round(conf, 2),
                 "actual": bundle["classes"][int(y_test.loc[idx])]})
pd.DataFrame(rows)